# 06 -- Experiment Results

Run representative experiments from the EigenDialectos suite, load results,
display key metrics, and compare across experiments.

**Key modules:** `experiments.runner`, `experiments.exp1_spectral_map`,
`experiments.exp3_dialectal_gradient`, `experiments.exp7_zeroshot`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from eigendialectos.constants import DialectCode, DIALECT_NAMES

## Experiment 1: Spectral Map

Eigendecompose all dialect transformation matrices, compute spectral entropies,
and build a pairwise distance matrix.

In [ ]:
from eigendialectos.experiments.exp1_spectral_map import SpectralMapExperiment

exp1 = SpectralMapExperiment()
exp1.setup({"dim": 30, "vocab_size": 100, "seed": 42})
result1 = exp1.run()

print(f"Experiment: {result1.experiment_id}")
print(f"Metrics keys: {list(result1.metrics.keys())}")
print(f"Mean distance: {result1.metrics['mean_distance']:.4f}")
print(f"Max distance:  {result1.metrics['max_distance']:.4f}")

In [ ]:
# Evaluate: do known-close pairs have smaller distances than known-distant pairs?
eval1 = exp1.evaluate(result1)
print("Evaluation Metrics:")
for k, v in eval1.items():
    print(f"  {k}: {v}")

In [ ]:
# Entropy rankings
entropies = result1.metrics["entropies"]
sorted_ent = sorted(entropies.items(), key=lambda x: x[1], reverse=True)

fig, ax = plt.subplots(figsize=(10, 5))
codes = [c for c, _ in sorted_ent]
vals = [v for _, v in sorted_ent]
ax.barh(codes, vals, color="teal")
ax.set_xlabel("Spectral Entropy")
ax.set_title("Experiment 1: Dialectal Spectral Entropy")
ax.axvline(np.mean(vals), color="red", linestyle="--", label=f"Mean={np.mean(vals):.3f}")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Distance heatmap
dist = np.array(result1.metrics["distance_matrix"])
order = result1.metrics["dialect_order"]

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(dist, cmap="YlOrRd")
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, rotation=45, ha="right")
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order)
fig.colorbar(im, ax=ax, label="Spectral Distance")
ax.set_title("Experiment 1: Pairwise Spectral Distance")
plt.tight_layout()
plt.show()

## Experiment 2: Full Dialect Generation

Apply DIAL at alpha=1.0 for every dialect and evaluate against reference samples.

In [ ]:
from eigendialectos.experiments.exp2_full_generation import FullGenerationExperiment

exp2 = FullGenerationExperiment()
exp2.setup({"dim": 30, "vocab_size": 100, "seed": 42})
result2 = exp2.run()

print(f"Experiment: {result2.experiment_id}")
print(f"Metrics keys: {list(result2.metrics.keys())}")

eval2 = exp2.evaluate(result2)
print("\nEvaluation:")
for k, v in eval2.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

## Experiment 3: Dialectal Gradient

Sweep alpha from 0.0 to 1.5 for each dialect and identify recognition and naturalness thresholds.

In [ ]:
from eigendialectos.experiments.exp3_dialectal_gradient import DialectalGradientExperiment

exp3 = DialectalGradientExperiment()
exp3.setup({"dim": 30, "vocab_size": 100, "seed": 42, "n_sentences": 20})
result3 = exp3.run()

eval3 = exp3.evaluate(result3)
print("Recognition thresholds by dialect:")
for code, val in sorted(eval3.get("recognition_thresholds", {}).items()):
    print(f"  {code}: alpha = {val}")

print(f"\nMean recognition threshold: {eval3.get('mean_recognition_threshold')}")
print(f"Mean naturalness threshold: {eval3.get('mean_naturalness_threshold')}")

In [ ]:
# Plot gradient curves for all dialects
curves = result3.metrics["curves"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for code in sorted(curves.keys()):
    data = curves[code]
    ax1.plot(data["alphas"], data["confidences"], label=code, linewidth=1.5)
    ax2.plot(data["alphas"], data["norms"], label=code, linewidth=1.5)

ax1.axhline(0.5, color="gray", linestyle="--", alpha=0.6, label="conf=0.5")
ax1.set_xlabel("Alpha")
ax1.set_ylabel("Classifier Confidence (proxy)")
ax1.set_title("Experiment 3: Alpha vs Confidence")
ax1.legend(fontsize=7, ncol=2)

ax2.set_xlabel("Alpha")
ax2.set_ylabel("Mean Embedding Norm")
ax2.set_title("Experiment 3: Alpha vs Embedding Norm")
ax2.legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.show()

## Experiment 4: Impossible Dialects

Create impossible dialect mixtures (contradictory features) and measure coherence degradation.

In [ ]:
from eigendialectos.experiments.exp4_impossible_dialects import ImpossibleDialectsExperiment

exp4 = ImpossibleDialectsExperiment()
exp4.setup({"dim": 30, "vocab_size": 100, "seed": 42})
result4 = exp4.run()

eval4 = exp4.evaluate(result4)
print(f"Experiment: {result4.experiment_id}")
print("\nEvaluation:")
for k, v in eval4.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

## Experiment 7: Zero-shot Transfer

Leave-2-out tensor completion: hold out 2 dialect varieties, reconstruct from
the remaining via tensor decomposition, and measure Frobenius error.

In [ ]:
from eigendialectos.experiments.exp7_zeroshot import ZeroshotTransferExperiment

exp7 = ZeroshotTransferExperiment()
exp7.setup({"dim": 15, "vocab_size": 60, "seed": 42})
result7 = exp7.run()

eval7 = exp7.evaluate(result7)
print(f"Experiment: {result7.experiment_id}")
print("\nEvaluation:")
for k, v in eval7.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6f}")
    elif isinstance(v, dict):
        print(f"  {k}:")
        for kk, vv in v.items():
            print(f"    {kk}: {vv}")
    else:
        print(f"  {k}: {v}")

## Cross-Experiment Summary

Aggregate key metrics from all experiments run above.

In [ ]:
results = {
    result1.experiment_id: result1,
    result2.experiment_id: result2,
    result3.experiment_id: result3,
    result4.experiment_id: result4,
    result7.experiment_id: result7,
}

print(f"{'Experiment':<30} {'Num Metrics':>12} {'Artifacts':>10}")
print("-" * 55)
for exp_id, res in sorted(results.items()):
    n_metrics = len(res.metrics)
    n_artifacts = len(res.artifact_paths)
    print(f"{exp_id:<30} {n_metrics:>12} {n_artifacts:>10}")

In [ ]:
# Generate textual reports
for exp, res in [(exp1, result1), (exp3, result3)]:
    report = exp.report(res)
    print(report[:500])
    print("...\n" + "=" * 60 + "\n")

## Using the ExperimentRunner

The `ExperimentRunner` provides an orchestrated lifecycle for running all
experiments with a shared config. Below we demonstrate its API without
running all 7 experiments (which may be slow).

In [ ]:
from eigendialectos.experiments.runner import ExperimentRunner

runner = ExperimentRunner(
    config={"dim": 20, "seed": 42},
    data_dir=Path("../data"),
    output_dir=Path("../outputs/experiments"),
)

print("Registered experiments:")
for exp_id in runner.list_experiments():
    print(f"  - {exp_id}")